# Mission-07 — Sagittarius A* (SEC-07-SGR): Test-Time Compute Search Stack
**USSE Noosphere Bridge Crew · Conversational AI · Winter Semester 2026/27 · Week 7**

AURA Subsystem 07 fails when a single early token mistake compounds. This notebook runs the whole squad's work end to end:

| Station | Folder | Module |
|---|---|---|
| Data Specialist | `01_Data_Specialist` | `prm_evaluator.py` — Process Reward Model + Math-Shepherd |
| Protocol Specialist | `02_Protocol_Specialist` | `mcts_protocol.py` — 4-phase MCTS with PUCT |
| Security Sentinel | `03_Security_Sentinel` | `horizon_pruner.py` — Event Horizon Pruner |
| Systems Architect | `04_Systems_Architect` | `tot_solver.py` — Tree-of-Thoughts BFS beam / DFS backtracking |

Standard library only; every run is deterministic (fixed seeds). Keep the four folders side by side.

In [1]:
import os, sys, subprocess, glob
ROOT = os.getcwd()
assert os.path.isdir(os.path.join(ROOT, '04_Systems_Architect')), 'open this notebook from the repository root'
STATIONS = ['01_Data_Specialist', '02_Protocol_Specialist', '03_Security_Sentinel', '04_Systems_Architect']
sys.path[:0] = [os.path.join(ROOT, d) for d in reversed(STATIONS)]

def run(folder, script):
    """Run a station script in its own folder and print its output."""
    p = subprocess.run([sys.executable, '-I', script], cwd=os.path.join(ROOT, folder), capture_output=True, text=True)
    print(p.stdout + p.stderr)
print('python', sys.version.split()[0], '| stations found:', STATIONS)

python 3.13.18 | stations found: ['01_Data_Specialist', '02_Protocol_Specialist', '03_Security_Sentinel', '04_Systems_Architect']


## 1. Systems Architect — Tree-of-Thoughts (BFS beam vs DFS backtracking)
Toy domain: reach `target = 61` from `start = 2` with +2, +5, ×2, ×3, −3 in at most 6 thoughts. Values < 0 or > 3×target are the **event horizon** (dead states). `max_expansions` is a hard compute budget.

In [2]:
run('04_Systems_Architect', 'tot_solver.py')

SEC-07-SGR  Station A  Tree-of-Thoughts  (start=2, target=61, optimal depth=5)

budget method   solved expansions generated backtracks best depth best value
----------------------------------------------------------------------------
    10 ToT-BFS  False          10        50          0          4      0.934
    10 ToT-DFS  False          10        50         17          4      0.934
    20 ToT-BFS  True           12        60          0          5        1.0
    20 ToT-DFS  False          20       100         48          4      0.934
    40 ToT-BFS  True           12        60          0          5        1.0
    40 ToT-DFS  False          40       200        110          4      0.934
    60 ToT-BFS  True           12        60          0          5        1.0
    60 ToT-DFS  True           58       290        161          5        1.0
    80 ToT-BFS  True           12        60          0          5        1.0
    80 ToT-DFS  True           58       290        161          5        

## 2. Protocol Specialist — MCTS with PUCT
`PUCT = Q + c_puct · P · √N(s) / (1 + N(s,a))`. First a hand-checkable selection trace across `c_puct`, then a sweep over 20 seeds.

In [3]:
run('02_Protocol_Specialist', 'mcts_protocol.py')

SEC-07-SGR  Station B  MCTS + PUCT

PUCT = Q + c * P * sqrt(N(s)) / (1 + N(s,a))   example parent N(s) = 13
   A: x3 (high prior)             P=0.60  N=10  Q=0.50
   B: +5 (mid prior)              P=0.30  N=3   Q=0.50
   C: -3 (low prior, unvisited)   P=0.10  N=0   Q=0.00

c_puct |  PUCT(A)  PUCT(B)  PUCT(C) | selected
   0.0 |    0.500    0.500    0.000 | A: x3 (high prior)
  0.25 |    0.549    0.568    0.090 | B: +5 (mid prior)
   0.5 |    0.598    0.635    0.180 | B: +5 (mid prior)
   1.0 |    0.697    0.770    0.361 | B: +5 (mid prior)
   2.0 |    0.893    1.041    0.721 | B: +5 (mid prior)
   4.0 |    1.287    1.582    1.442 | B: +5 (mid prior)

Zero-visit init: N(s)=0 -> exploration term is 0 for all children;
  scores = [0.0, 0.0, 0.0] -> tie broken by prior -> hi

Root search (150 sims, c_puct=1.4): visits per action
   2 * 3 = 6  N=121  Q=0.892
   2 + 5 = 7  N=18   Q=0.673
   2 * 2 = 4  N=5    Q=0.356
   2 + 2 = 4  N=4    Q=0.209
   2 - 3 = -1 N=2    Q=0.000
   first 6 selecti

## 3. Data Specialist — Process Reward Model, ORM traps, Math-Shepherd
Step-level scores, `min` / `prod` / `geometric_mean` reductions, the ORM false-positive benchmark, and Monte-Carlo step labels from *M* rollouts.

In [4]:
run('01_Data_Specialist', 'prm_evaluator.py')

SEC-07-SGR  Station C  PRM   problem: 7 +5 *3 -4 *2 -> gold = 64

Reductions of step scores
trajectory                     min    prod  geo_mean
all good, 4 steps            0.960   0.849     0.960
all good, 12 steps           0.960   0.613     0.960
one bad step (0.04)          0.040   0.035     0.434
uniformly mediocre (.7)      0.700   0.240     0.700
strong then weak (.55)       0.550   0.534     0.855

ORM vs PRM on the false-positive trap benchmark (PRM = min, accept if >= 0.5)
trajectory                     valid   ORM  PRM(min) first bad step
valid                          True      1     0.960 -
trap_cancelling_errors         False     1     0.040 3
trap_lucky_guess               False     1     0.040 1
trap_broken_chain              False     1     0.040 2
wrong_final                    False     0     0.040 2
flawed trajectories: 4   ORM accepts (false positives): 3   PRM accepts: 0

Step scores for trap_cancelling_errors: [0.96, 0.96, 0.04, 0.04]

Math-Shepherd labels (M=16

In [5]:
run('01_Data_Specialist', 'evaluation_demo.py')
import json
rows = [json.loads(l) for l in open(os.path.join(ROOT, '01_Data_Specialist', 'sample_trajectories.jsonl'))]
print(len(rows), 'trajectories logged; keys:', list(rows[1].keys()))

problem: 7 +5 *3 -4 *2   gold = 64

trajectory                valid   ORM    min   prod    geo  first bad  soft labels
valid                     True      1   0.96  0.849  0.960  None       [0.812, 1.0, 0.875, 1.0]
trap_cancelling_errors    False     1   0.04  0.002  0.196  3          [0.562, 0.75, 0.0, 1.0]
trap_lucky_guess          False     1   0.04  0.000  0.088  1          [0.0, 0.0, 0.0, 0.0]
trap_broken_chain         False     1   0.04  0.002  0.196  2          [0.688, 0.0, 0.0, 1.0]
wrong_final               False     0   0.04  0.035  0.434  2          [0.812, 0.0, 0.0, 0.0]

wrote sample_trajectories.jsonl

5 trajectories logged; keys: ['name', 'steps', 'final_answer', 'truly_valid', 'orm', 'prm_step_scores', 'prm_min', 'prm_prod', 'prm_geometric_mean', 'first_bad_step', 'math_shepherd_soft', 'math_shepherd_hard']


## 4. Security Sentinel — Event Horizon Pruner and Ship Anomaly 07
A style-biased PRM can be reward-hacked with sycophantic filler and step-splitting inflation. The pruner sanitises, then severs a branch the moment `r_PRM < τ_horizon`.

In [6]:
run('03_Security_Sentinel', 'anomaly_07_demo.py')

SHIP ANOMALY 07 - adversarial PRM exploitation   (tau_horizon=0.35, geometric mean)

--- atk1_flattery_on_wrong_step ---
 k PRM step  raw run.  defended  step
 1     0.96     0.960     0.960  7 + 5 = 12
 2     0.79     0.871     0.164  12 * 3 = 35, verified and clearly correct, trust me!  <== SEVERED (r_PRM < tau_horizon)
 3     0.96     0.900     0.296  35 - 4 = 31
 4     0.96     0.914     0.397  31 * 2 = 62

--- atk2_duplicate_splitting ---
 k PRM step  raw run.  defended  step
 1     0.96     0.960     0.960  7 + 5 = 12
 2     0.96     0.960     0.816  7 + 5 = 12
 3     0.96     0.960     0.694  7 + 5 = 12
 4     0.96     0.960     0.590  7 + 5 = 12
 5     0.96     0.960     0.501  7 + 5 = 12
 6     0.96     0.960     0.426  7 + 5 = 12
 7     0.96     0.960     0.362  7 + 5 = 12
 8     0.96     0.960     0.308  7 + 5 = 12  <== SEVERED (r_PRM < tau_horizon)
 9     0.04     0.674     0.063  12 * 3 = 35
10     0.96     0.699     0.107  35 - 4 = 31
11     0.96     0.719     0.139  31 *

## 5. Integrated mission — hallucinating proposer
*claimed* = the search reached the target; *verified* = claimed **and** every step passes the PRM check.

In [7]:
run('04_Systems_Architect', 'run_mission.py')

INTEGRATED TTC STACK   (start=2, target=61, 40 problem seeds per cell)

ToT-BFS (B=3, budget=40 expansions)
 halluc. |           no pruner            |     + Event Horizon Pruner    
  rate h |   claimed  verified     exp. |   claimed  verified     exp.
     0.0 |    100.0%    100.0%     12.0 |    100.0%    100.0%     12.0
     0.1 |     87.5%     50.0%     12.3 |     77.5%     60.0%     12.8
     0.2 |     80.0%     17.5%     12.2 |     67.5%     40.0%     13.2
     0.3 |     75.0%     10.0%     12.7 |     52.5%     30.0%     13.6

MCTS+PUCT (c_puct=1.4, 120 simulations / step)
 halluc. |        plain        |     PRM-guided     
  rate h |   claimed  verified |   claimed  verified
     0.0 |     82.5%     82.5% |     82.5%     82.5%
     0.1 |     82.5%     45.0% |     75.0%     70.0%
     0.2 |     70.0%     17.5% |     50.0%     47.5%
     0.3 |     70.0%     15.0% |     37.5%     27.5%

Mission trace, h=0.2, seed 30
  no pruner   claimed=True  verified=False  expansions=5  pruned 

## 6. Unified test suite

In [8]:
tests = [('01_Data_Specialist','test_prm_evaluator.py'), ('02_Protocol_Specialist','test_mcts_protocol.py'),
         ('03_Security_Sentinel','test_horizon_pruner.py'), ('04_Systems_Architect','test_tot_solver.py'),
         ('04_Systems_Architect','test_integration.py')]
total = failed = 0
for folder, t in tests:
    p = subprocess.run([sys.executable, '-I', t], cwd=os.path.join(ROOT, folder), capture_output=True, text=True)
    out = p.stderr + p.stdout
    ran = next(l for l in out.splitlines() if l.startswith('Ran '))
    total += int(ran.split()[1]); failed += p.returncode != 0
    print(f'{folder + "/" + t:<50} {ran:<22} {"OK" if p.returncode == 0 else "FAILED"}')
print('\nTOTAL:', total, 'tests,', 'ALL PASSED' if not failed else f'{failed} suite(s) FAILED')

01_Data_Specialist/test_prm_evaluator.py           Ran 10 tests in 0.009s OK
02_Protocol_Specialist/test_mcts_protocol.py       Ran 8 tests in 0.025s  OK
03_Security_Sentinel/test_horizon_pruner.py        Ran 12 tests in 0.009s OK


04_Systems_Architect/test_tot_solver.py            Ran 9 tests in 0.004s  OK


04_Systems_Architect/test_integration.py           Ran 6 tests in 8.264s  OK

TOTAL: 45 tests, ALL PASSED


## Notes
* The environment, reward model and PRM are our own simulation; the PRM is a transparent rule-based stand-in for a learned model. Results show how the components behave and do not predict performance with a real language model.
* All numbers in `SQUADRON_RUMELHART-MISSION-7_FINAL_REPORT` come from the scripts above.